## Online Payment Fraud Detection (EDA + Modeling)

This notebook performs:
- Basic EDA (class balance, summary stats, plots)
- Train/test split
- SMOTE to handle class imbalance
- Train **Logistic Regression** and **Random Forest**
- Compare using **F1-score**
- Confusion matrix heatmap
- Save best model to `models/fraud_model.pkl`


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Make src/ importable
import sys
ROOT = Path.cwd().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    FEATURE_COLUMNS,
    coerce_schema,
    find_target_column,
    load_dataset,
    make_preprocessor,
    project_paths,
)

paths = project_paths()
paths

In [ ]:
# Load dataset (or generate synthetic if missing)
df_raw = load_dataset(paths.data_csv)
df = coerce_schema(df_raw)
target_col = find_target_column(df)

print('Shape:', df.shape)
print('Target column:', target_col)

# Standardize target for analysis
if target_col != 'isFraud':
    df['isFraud'] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)

# Basic stats
print(df[FEATURE_COLUMNS + ['isFraud']].describe(include='all'))

# Class distribution
plt.figure(figsize=(5,3))
sns.countplot(data=df, x='isFraud')
plt.title('Class distribution (0=Legit, 1=Fraud)')
plt.tight_layout()
plt.show()

df['isFraud'].value_counts(normalize=True)

In [ ]:
# Quick EDA plots
plt.figure(figsize=(6,3))
sns.histplot(df['amount'].dropna(), bins=50, kde=True)
plt.title('Transaction amount distribution')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,3))
sns.boxplot(data=df, x='isFraud', y='amount')
plt.title('Amount by class')
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
X = df[FEATURE_COLUMNS]
y = df['isFraud'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

pre = make_preprocessor()

# Model 1: Logistic Regression (with SMOTE)
log_reg = LogisticRegression(max_iter=2000, n_jobs=None)
pipe_lr = ImbPipeline([
    ('preprocess', pre),
    ('smote', SMOTE(random_state=42)),
    ('model', log_reg),
])

# Model 2: Random Forest (with SMOTE)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
pipe_rf = ImbPipeline([
    ('preprocess', pre),
    ('smote', SMOTE(random_state=42)),
    ('model', rf),
])

models = {
    'LogisticRegression': pipe_lr,
    'RandomForest': pipe_rf,
}

scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    f1 = f1_score(y_test, pred, zero_division=0)
    scores[name] = f1
    print(f'{name} F1-score: {f1:.4f}')

scores

In [ ]:
# Confusion matrix for best model
best_name = max(scores, key=scores.get)
best_model = models[best_name]

pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred, zero_division=0))

# Save best model
paths.model_pkl.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, paths.model_pkl)
print('Saved best model to', paths.model_pkl)